In [ ]:
## Stochastic Simulation - Day 3

# Write a discrete event simulation program for a blocking system, i.e. a system with m service units and no waitning room.
# The offered traffic A is the product of the mean arrival rate and the mean service time. 

In [ ]:
# Exercise 4 - Part 1

import numpy as np
from scipy.stats import t
from math import factorial


M = 10                       # number of service units
MEAN_INTERARRIVAL = 1.0      # mean time between customers
MEAN_SERVICE = 8.0           # mean service time
ARRIVAL_RATE = 1 / MEAN_INTERARRIVAL
SERVICE_RATE = 1 / MEAN_SERVICE
N_CUSTOMERS = 10000
N_REPLICATIONS = 10


class BlockingSystem:
    """
    
    Blocking system with m servers and no waiting room.
     - The arrival process is modeled as a Poisson process --> inter-arrival times are independent and exponentially distributed.
     - The service time distribution is exponential --> meaning we are drawing from an exponential distribution for each service time.

    STATE VARIABLES:
     - busy: number of busy service units
     - arrivals: total number of arriving customers
     - blocked: total number of blocked customers

    """

    def __init__(self):
        self.busy = 0              # number of busy service units
        self.arrivals = 0          # total arriving customers
        self.blocked = 0           # blocked customers

    def events(self):
        """Return event rates: arrival and departure."""
        return [
            ARRIVAL_RATE,                # 0: arrival
            SERVICE_RATE * self.busy     # 1: departure (after service completion)
        ]

    def adjust_system(self, eventnumber):
        """Update the system according to the event."""

        if eventnumber == 0:
            # A new customer arrives
            self.arrivals += 1

            if self.busy < M:
                # At least one service unit is free
                self.busy += 1
            else:
                # All service units are busy, so the customer is blocked
                self.blocked += 1

        elif eventnumber == 1:
            # A customer finishes service
            if self.busy > 0:
                self.busy -= 1

    def blocking_fraction(self):
        """Fraction of customers that were blocked."""
        return self.blocked / self.arrivals

    def run_simulation(n_customers=N_CUSTOMERS):
        """
        
        Run one simulation until n_customers have arrived.
        
        One iteration does:
        1. Determine current rates
        2. Draw time until next event
        3. Advance clock
        4. Decide event type
        5. Update system state 
        """

        model = BlockingSystem()
        t_now = 0

        while model.arrivals < n_customers: #continue until 10 000 costumers have arrived
            rates = model.events() # a list of the rates for the two events
            a0 = sum(rates) # the total rate at which anything happens

            # Time to next event
            dt = np.random.exponential(1 / a0) #waiting time until any evet happens is exponentially distributed with mean 1/a0
            t_now += dt # jumping to the time of the next event

            # Choose which event happens
            probs = np.array(rates) / a0 # probability of each event 
            event = np.random.choice(len(rates), p=probs) # drawing the event that happens according to the probabilities

            # Update system
            model.adjust_system(event)

        return model.blocking_fraction()



def run_replications(n_replications=N_REPLICATIONS):
    """Run several independent simulations and compute confidence interval."""

    estimates = []

    for i in range(n_replications):
        estimate = run_simulation()
        estimates.append(estimate)

    estimates = np.array(estimates)

    mean_estimate = np.mean(estimates)
    sample_sd = np.std(estimates, ddof=1)

    alpha = 0.05
    critical_value = t.ppf(1 - alpha / 2, df=n_replications - 1) # nunber of standard errors away from the mean to get a 95% confidence interval

    half_width = critical_value * sample_sd / np.sqrt(n_replications) # the actual distance away from the mean. Often the number denoted after plus/minus sign in an estimate.

    ci = (
        float(mean_estimate - half_width),
        float(mean_estimate + half_width)
    )

    return mean_estimate, ci, estimates


def erlang_B(m=M, A=ARRIVAL_RATE * MEAN_SERVICE):
    """Exact Erlang B blocking probability."""

    numerator = A**m / factorial(m)
    denominator = sum(A**i / factorial(i) for i in range(m + 1))

    return numerator / denominator


# Run exercise
mean_blocking, ci, estimates = run_replications()
exact = erlang_B()

print("Simulation estimates:", estimates)
print("Estimated blocking fraction:", mean_blocking)
print("95% confidence interval:", ci)
print("Exact Erlang B value:", exact)

Simulation estimates: [0.1157 0.127  0.1203 0.1212 0.1256 0.1134 0.1223 0.1233 0.1207 0.1294]
Estimated blocking fraction: 0.12188999999999998
95% confidence interval: (0.11840786967715382, 0.12537213032284616)
Exact Erlang B value: 0.12166106425295149


In [37]:
## Exercise 4 - Part 1: SIMPLER IMPLEMENTATION

import numpy as np
from math import factorial

def block_system_poisson(n_customers, n_service_units, t_mean_service, t_mean_customers):

    t_res = []
    blocked_frac_res = []
    departures_res =  []

    for _ in range(n_service_units):
        # State variables
        departure_times = []        # List of departure times for custormers currently being serviced 
        number_of_blocked = 0    
               
        t = 0                       
        t_arrival = 0               

        # Loop through all customers
        for _ in range(int(n_customers)):

            t += t_arrival # jumping to next event

            #Update departure list if not empty: keep only customers in list, whose departure time > current time
            if len(departure_times) != 0:
                departure_times = [dep for dep in departure_times if dep > t]

            #Check whether capacity is full. If full, block customer
            if len(departure_times) >= 10:
                number_of_blocked += 1                                     
            #If not full, assign service time and add customer to departure list
            else:
                #Assign service time for current customer
                t_service = np.random.exponential(t_mean_service)     
                #Add customers departure time to departure_times list
                departure_times.append(t+t_service)

            # Arrival time for next costumer
            t_arrival = np.random.exponential(t_mean_customers)
        
        # Storing results for every service unit
        t_res.append(t)
        blocked_frac_res.append(number_of_blocked/n_customers)
        departures_res.append(departure_times)

    return np.array(blocked_frac_res), np.array(t_res), departures_res

# Run simulation
blocked_frac_res, t_res, departures_res = block_system_poisson(
                                                                n_customers=1*1e4,
                                                                n_service_units=10, 
                                                                t_mean_service=8, 
                                                                t_mean_customers=1
                                                                )


def erlang_B(m= 10, A= (1/1) * 8):
    """Exact Erlang B blocking probability."""

    numerator = A**m / factorial(m)
    denominator = sum(A**i / factorial(i) for i in range(m + 1))

    return numerator / denominator

exact = erlang_B()

print("Blocked fraction:")
print(blocked_frac_res)
print("Exact Erlang B value:")
print(exact)
print("Total time:")
print(t_res)
print("Departure times:")
print(departures_res)
print("Length of departures_res:")
print(len(departures_res))
print("Length of each element in departures_res:")
for x in departures_res:
    print(len(x)) # each entry gives the number of costumers for that service unit still being serviced at the final simulation time.

Blocked fraction:
[0.1179 0.1304 0.1203 0.1246 0.1141 0.1142 0.1251 0.1265 0.13   0.124 ]
Exact Erlang B value:
0.1216610642529515
Total time:
[ 9989.51053056  9856.4393478  10054.47003122  9979.89778678
  9945.46965847 10082.83838531  9916.83462766  9992.528743
  9984.17204656  9941.58754074]
Departure times:
[[9999.79287111658, 10002.328477308007, 9990.590301825127, 9993.811930885371, 10018.275365921361, 9993.707323197395, 10004.158494930576], [9877.70069469518, 9866.781234423876, 9873.680482659445, 9868.205808237351, 9856.714369314037, 9875.331142479112, 9881.748104828952, 9860.790260042422], [10054.768476380685, 10055.127909510771, 10064.23229932513, 10056.008113680626, 10064.141273160014, 10055.307845963487, 10065.296621752937, 10058.155593086867], [9980.849782261535, 9982.286547351956, 9980.925410367361, 9987.453912771423, 9982.994073309523], [9949.334261481894, 9950.968701924583, 9957.167760050113, 9962.653299384057, 9945.830604460587, 9947.626634496342, 9964.61272143378, 9945.6

In [38]:
# Exercise 4 - Part 2

## Changing the distribution for the inter arrival time

import numpy as np

def block_system_poisson(n_customers, n_service_units, t_mean_service, t_mean_customers):

    t_res = []
    blocked_frac_res = []
    departures_res =  []

    for _ in range(n_service_units):
        # State variables
        departure_times = []        # List of departure times for custormers currently being serviced 
        number_of_blocked = 0    
               
        t = 0                       
        t_arrival = 0               

        # Loop through all customers
        for _ in range(int(n_customers)):

            t += t_arrival # jumping to next event

            #Update departure list if not empty: keep only customers in list, whose departure time > current time
            if len(departure_times) != 0:
                departure_times = [dep for dep in departure_times if dep > t]

            #Check whether capacity is full. If full, block customer
            if len(departure_times) >= 10:
                number_of_blocked += 1                                     
            #If not full, assign service time and add customer to departure list
            else:
                #Assign service time for current customer
                t_service = np.random.exponential(t_mean_service)     
                #Add customers departure time to departure_times list
                departure_times.append(t+t_service)

            # Arrival time for next costumer

            # ERLANG
            #t_arrival = np.random.gamma(shape=2, scale=1/2)

            # HYPEREXPONENTIAL
            u = np.random.rand()
            if u < 0.8:
                # distribution 1: Exp(lambda_1 = 0.8333)
                t_arrival = np.random.exponential(scale=1/0.8333)
            else:
                # distribution 2: Exp(lambda_2 = 5.0)
                t_arrival = np.random.exponential(scale=1/5.0)

        
        # Storing results for every service unit
        t_res.append(t)
        blocked_frac_res.append(number_of_blocked/n_customers)
        departures_res.append(departure_times)

    return np.array(blocked_frac_res), np.array(t_res), departures_res

# Run simulation
blocked_frac_res, t_res, departures_res = block_system_poisson(
                                                                n_customers=1*1e4,
                                                                n_service_units=10, 
                                                                t_mean_service=8, 
                                                                t_mean_customers=1
                                                                )


print("Blocked fraction:")
print(blocked_frac_res)

mean_block = np.mean(blocked_frac_res)
std_block = np.std(blocked_frac_res, ddof=1)
n = len(blocked_frac_res)

ci_low = mean_block - 1.96 * std_block / np.sqrt(n)
ci_high = mean_block + 1.96 * std_block / np.sqrt(n)

print("Mean blocked fraction:", mean_block)
print("95% CI:", ci_low, ci_high)


Blocked fraction:
[0.1361 0.1401 0.1422 0.1375 0.1372 0.1383 0.1332 0.1423 0.1442 0.1351]
Mean blocked fraction: 0.13862000000000002
95% CI: 0.13644158640392706 0.14079841359607298


In [39]:
# Exercise 4 - Part 3

# Different service time distribution

## Exercise 4 - Part 1: SIMPLER IMPLEMENTATION

import numpy as np

def block_system_poisson(n_customers, n_service_units, t_mean_service, t_mean_customers):

    t_res = []
    blocked_frac_res = []
    departures_res =  []

    for _ in range(n_service_units):
        # State variables
        departure_times = []        # List of departure times for custormers currently being serviced 
        number_of_blocked = 0    
               
        t = 0                       
        t_arrival = 0               

        # Loop through all customers
        for _ in range(int(n_customers)):

            t += t_arrival # jumping to next event

            #Update departure list if not empty: keep only customers in list, whose departure time > current time
            if len(departure_times) != 0:
                departure_times = [dep for dep in departure_times if dep > t]

            #Check whether capacity is full. If full, block customer
            if len(departure_times) >= 10:
                number_of_blocked += 1                                     
            #If not full, assign service time and add customer to departure list
            else:
                #Assign service time for current customer

                # CONSTANT SERVICE TIME
                t_service = t_mean_service

                # PARETO SERVICE TIME
                #alpha = 1.05 # 2.05
                #mean_service = t_mean_service
                #scale = (alpha - 1) / alpha * mean_service
                #t_service = scale * (1 + np.random.pareto(alpha))

                # ERLANG
                #t_service = np.random.gamma(shape=2, scale=1/2)
    
                #Add customers departure time to departure_times list
                departure_times.append(t+t_service)

            # Arrival time for next costumer
            t_arrival = np.random.exponential(t_mean_customers)
        
        # Storing results for every service unit
        t_res.append(t)
        blocked_frac_res.append(number_of_blocked/n_customers)
        departures_res.append(departure_times)

    return np.array(blocked_frac_res), np.array(t_res), departures_res

# Run simulation
blocked_frac_res, t_res, departures_res = block_system_poisson(
                                                                n_customers=1*1e4,
                                                                n_service_units=10, 
                                                                t_mean_service=8, 
                                                                t_mean_customers=1
                                                                )


print("Blocked fraction:")
print(blocked_frac_res)

mean_block = np.mean(blocked_frac_res)
std_block = np.std(blocked_frac_res, ddof=1)
n = len(blocked_frac_res)

ci_low = mean_block - 1.96 * std_block / np.sqrt(n)
ci_high = mean_block + 1.96 * std_block / np.sqrt(n)

print("Mean blocked fraction:", mean_block)
print("95% CI:", ci_low, ci_high)


Blocked fraction:
[0.1233 0.123  0.1229 0.1234 0.1224 0.1207 0.1217 0.1187 0.1171 0.1125]
Mean blocked fraction: 0.12057
95% CI: 0.11838254282175247 0.12275745717824753
